In [38]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import solve_ivp

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (9, 5), "font.size": 11})

BLUE = "#006DAE"
ORANGE = "#E57200"
GREEN = "#00843D"
RED = "#C8102E"
PURPLE = "#7A52A1"

# Conditions for Blasius
INITIAL_f = 0.0
INITIAL_df = 0.0
FINAL_df = 1.0
ETA_0 = 0.0
ETA_MAX = 10.0
GUESS_ddF = 1.0
GUESS2_ddF = 20.0

# Constants
CONSTANT_K = np.sqrt(2)
CONSTANT_C = np.sqrt(2)

# Step size
STEP_SIZE = 0.1/CONSTANT_C
ETA_ARRAY = np.arange(ETA_0, ETA_MAX, float(STEP_SIZE))

# Blasius ODE initial condition
def blasius_rhs(eta, blasius):
    '''Input is the scaled submerge distance and the Blasius vector as numpy array 1x3
    Output is the scaled Blasius vector

    Returns d/dn where n is the scaled submerge distance,
    Return is in the form of [dF, ddF, and dddF], and dimensionless'''

    # Input is the scaled submerge distance and the Blasius vector as numpy array 1x3
    # Output is the scaled Blasius vector
    # Blasius is as:
    # [F, dF, ddF] where:
    # F is the scaled stream function
    # dF is u/U, i.e., velocity along the board as a fraction of the free-stream speed
    # ddf is the scaled velocity gradient function - i.e., shear
    f = blasius[0]
    df = blasius[1]
    ddf = blasius[2]

    return np.array([
        df,
        ddf,
        -(1/2)*f*ddf
    ])


# extracting data from csv
white_blasius_pd = pd.read_csv('blasius_table_white_majdalani_4e.csv', comment='#')
white_blasius = np.asarray(white_blasius_pd)
white_eta = white_blasius[:, 0]*CONSTANT_C
white_f = white_blasius[:, 2]*CONSTANT_K
white_df = white_blasius[:, 3]*CONSTANT_K/CONSTANT_C
white_ddf = white_blasius[:, 4]*CONSTANT_K/CONSTANT_C**2
white_dddf = white_blasius[:, 5]*CONSTANT_K/CONSTANT_C**3


def exact_blasius(eta):
    '''Exact Blasius solution extracted from White & Majdalani Table 4-3.'''

    # Finding value within table through conversion
    distances = np.abs(white_eta - eta)
    if np.min(distances) > STEP_SIZE:
        raise ValueError("Submerge scaled is not in the table")
    index = np.argmin(distances)

    return white_f[index], white_df[index], white_ddf[index], white_dddf[index]

# Claude produced test for exact_blasius function


In [41]:
# Shooting method tolerance & max iterations
TOLERANCE = 1e-6
MAX_ITERATIONS = 1000

# Shooting method
def F(s):
    #RK4 from MMA3001 notebook 7.4
    BLASIUS_RK4 = np.zeros((len(ETA_ARRAY), 3))
    BLASIUS_RK4[0] = np.array([INITIAL_F, INITIAL_dF, s])

    # RK4 integration
    for i in range(len(ETA_ARRAY)-1):
        eta = ETA_ARRAY[i]
        k1 = blasius_rhs(eta, BLASIUS_RK4[i])
        BLASIUS_trial_2 = BLASIUS_RK4[i] + 0.5*STEP_SIZE*k1
        k2 = blasius_rhs(eta+0.5*STEP_SIZE, BLASIUS_trial_2)
        BLASIUS_trial_3 = BLASIUS_RK4[i] + 0.5*STEP_SIZE*k2
        k3 = blasius_rhs(eta+0.5*STEP_SIZE, BLASIUS_trial_3)
        BLASIUS_trial_4 = BLASIUS_RK4[i] + STEP_SIZE*k3
        k4 = blasius_rhs(eta+STEP_SIZE, BLASIUS_trial_4)
        WEIGHTED_RATE = (k1+2*k2+2*k3+k4)/6


        blasius_rk4_next = BLASIUS_RK4[i]+STEP_SIZE*WEIGHTED_RATE
        BLASIUS_RK4[i+1] = blasius_rk4_next
    return BLASIUS_RK4[-1][1]-1

    # Adding the secant method to help find the next guess

# Defense


In [37]:
testa = F(5)